In [1]:
import csv
import json
import time
from kafka import KafkaProducer

producer = KafkaProducer(
    bootstrap_servers='localhost:9092',
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)

csv_file = r'C:\Users\USER PC\bigdata_project\customer_behavior.csv'
topic = 'customer-behavior'
batch_size = 1000
count = 0

print("Starting Kafka Producer...")

with open(csv_file, 'r') as f:
    reader = csv.DictReader(f)
    for row in reader:
        producer.send(topic, value=row)
        count += 1
        if count % batch_size == 0:
            producer.flush()
            print(f"  Sent {count:,} records...")
            time.sleep(0.1)  # slight delay to simulate streaming

producer.flush()
print(f"\nDone! Total records sent: {count:,}")

Starting Kafka Producer...
  Sent 1,000 records...
  Sent 2,000 records...
  Sent 3,000 records...
  Sent 4,000 records...
  Sent 5,000 records...
  Sent 6,000 records...
  Sent 7,000 records...
  Sent 8,000 records...
  Sent 9,000 records...
  Sent 10,000 records...
  Sent 11,000 records...
  Sent 12,000 records...
  Sent 13,000 records...
  Sent 14,000 records...
  Sent 15,000 records...
  Sent 16,000 records...
  Sent 17,000 records...
  Sent 18,000 records...
  Sent 19,000 records...
  Sent 20,000 records...
  Sent 21,000 records...
  Sent 22,000 records...
  Sent 23,000 records...
  Sent 24,000 records...
  Sent 25,000 records...
  Sent 26,000 records...
  Sent 27,000 records...
  Sent 28,000 records...
  Sent 29,000 records...
  Sent 30,000 records...
  Sent 31,000 records...
  Sent 32,000 records...
  Sent 33,000 records...
  Sent 34,000 records...
  Sent 35,000 records...
  Sent 36,000 records...
  Sent 37,000 records...
  Sent 38,000 records...
  Sent 39,000 records...
  Sent 

In [2]:
import findspark, os
findspark.init(r'C:\spark')
os.makedirs(r'C:\Users\USER PC\bigdata_project\spark-tmp', exist_ok=True)

from pyspark.sql import SparkSession
from pyspark.sql.functions import from_json, col
from pyspark.sql.types import *

spark = SparkSession.builder \
    .appName("CustomerBehaviorBatch") \
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.8") \
    .config("spark.local.dir", r"C:\Users\USER PC\bigdata_project\spark-tmp") \
    .config("spark.sql.warehouse.dir", r"C:\Users\USER PC\bigdata_project\spark-warehouse") \
    .config("spark.hadoop.fs.defaultFS", "file:///") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

# Read as batch (not streaming) to confirm data exists
batch_df = spark.read \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "localhost:9092") \
    .option("subscribe", "customer-behavior") \
    .option("startingOffsets", "earliest") \
    .load()

print("Total records in Kafka:", batch_df.count())
batch_df.show(3, truncate=False)

Total records in Kafka: 3000000
+----+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [3]:
from pyspark.sql.functions import from_json, col
from pyspark.sql.types import *

schema = StructType([
    StructField("customer_id", StringType()),
    StructField("age", StringType()),
    StructField("session_duration_mins", StringType()),
    StructField("pages_visited", StringType()),
    StructField("clicks", StringType()),
    StructField("time_on_site_mins", StringType()),
    StructField("previous_purchases", StringType()),
    StructField("cart_additions", StringType()),
    StructField("device_type", StringType()),
    StructField("browser", StringType()),
    StructField("region", StringType()),
    StructField("membership_level", StringType()),
    StructField("discount_applied", StringType()),
    StructField("time_since_last_visit_days", StringType()),
    StructField("will_purchase", StringType())
])

# Parse JSON from Kafka
parsed_df = batch_df.select(
    from_json(col("value").cast("string"), schema).alias("data")
).select("data.*")

# Cast all numeric columns to correct types
processed_df = parsed_df \
    .withColumn("age", col("age").cast("int")) \
    .withColumn("session_duration_mins", col("session_duration_mins").cast("double")) \
    .withColumn("pages_visited", col("pages_visited").cast("int")) \
    .withColumn("clicks", col("clicks").cast("int")) \
    .withColumn("time_on_site_mins", col("time_on_site_mins").cast("double")) \
    .withColumn("previous_purchases", col("previous_purchases").cast("int")) \
    .withColumn("cart_additions", col("cart_additions").cast("int")) \
    .withColumn("discount_applied", col("discount_applied").cast("int")) \
    .withColumn("time_since_last_visit_days", col("time_since_last_visit_days").cast("int")) \
    .withColumn("will_purchase", col("will_purchase").cast("int"))

# Drop nulls
processed_df = processed_df.dropna()

print("Total records after processing:", processed_df.count())
processed_df.show(5)
processed_df.printSchema()

Total records after processing: 3000000
+--------------------+---+---------------------+-------------+------+-----------------+------------------+--------------+-----------+-------+-------+----------------+----------------+--------------------------+-------------+
|         customer_id|age|session_duration_mins|pages_visited|clicks|time_on_site_mins|previous_purchases|cart_additions|device_type|browser| region|membership_level|discount_applied|time_since_last_visit_days|will_purchase|
+--------------------+---+---------------------+-------------+------+-----------------+------------------+--------------+-----------+-------+-------+----------------+----------------+--------------------------+-------------+
|1bba3a55-6e93-46d...| 56|                 6.46|           14|   113|            25.32|                14|             7|     mobile|   Edge|   West|          silver|               0|                        54|            0|
|c2f9571a-d7ce-4d3...| 32|                 0.91|           1

In [4]:
from pyspark.sql.functions import from_json, col
from pyspark.sql.types import *

schema = StructType([
    StructField("customer_id", StringType()),
    StructField("age", StringType()),
    StructField("session_duration_mins", StringType()),
    StructField("pages_visited", StringType()),
    StructField("clicks", StringType()),
    StructField("time_on_site_mins", StringType()),
    StructField("previous_purchases", StringType()),
    StructField("cart_additions", StringType()),
    StructField("device_type", StringType()),
    StructField("browser", StringType()),
    StructField("region", StringType()),
    StructField("membership_level", StringType()),
    StructField("discount_applied", StringType()),
    StructField("time_since_last_visit_days", StringType()),
    StructField("will_purchase", StringType())
])

parsed_df = batch_df.select(
    from_json(col("value").cast("string"), schema).alias("data")
).select("data.*")

processed_df = parsed_df \
    .withColumn("age", col("age").cast("int")) \
    .withColumn("session_duration_mins", col("session_duration_mins").cast("double")) \
    .withColumn("pages_visited", col("pages_visited").cast("int")) \
    .withColumn("clicks", col("clicks").cast("int")) \
    .withColumn("time_on_site_mins", col("time_on_site_mins").cast("double")) \
    .withColumn("previous_purchases", col("previous_purchases").cast("int")) \
    .withColumn("cart_additions", col("cart_additions").cast("int")) \
    .withColumn("discount_applied", col("discount_applied").cast("int")) \
    .withColumn("time_since_last_visit_days", col("time_since_last_visit_days").cast("int")) \
    .withColumn("will_purchase", col("will_purchase").cast("int"))

processed_df = processed_df.dropna()

print("Total records after processing:", processed_df.count())
processed_df.show(5)
processed_df.printSchema()

Total records after processing: 3000000
+--------------------+---+---------------------+-------------+------+-----------------+------------------+--------------+-----------+-------+-------+----------------+----------------+--------------------------+-------------+
|         customer_id|age|session_duration_mins|pages_visited|clicks|time_on_site_mins|previous_purchases|cart_additions|device_type|browser| region|membership_level|discount_applied|time_since_last_visit_days|will_purchase|
+--------------------+---+---------------------+-------------+------+-----------------+------------------+--------------+-----------+-------+-------+----------------+----------------+--------------------------+-------------+
|1bba3a55-6e93-46d...| 56|                 6.46|           14|   113|            25.32|                14|             7|     mobile|   Edge|   West|          silver|               0|                        54|            0|
|c2f9571a-d7ce-4d3...| 32|                 0.91|           1

In [6]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator

device_indexer = StringIndexer(inputCol="device_type", outputCol="device_idx")
browser_indexer = StringIndexer(inputCol="browser", outputCol="browser_idx")
region_indexer = StringIndexer(inputCol="region", outputCol="region_idx")
membership_indexer = StringIndexer(inputCol="membership_level", outputCol="membership_idx")

assembler = VectorAssembler(
    inputCols=[
        "age", "session_duration_mins", "pages_visited", "clicks",
        "time_on_site_mins", "previous_purchases", "cart_additions",
        "discount_applied", "time_since_last_visit_days",
        "device_idx", "browser_idx", "region_idx", "membership_idx"
    ],
    outputCol="features"
)

lr = LogisticRegression(featuresCol="features", labelCol="will_purchase")

pipeline = Pipeline(stages=[
    device_indexer, browser_indexer, region_indexer, membership_indexer,
    assembler, lr
])

train_df, test_df = processed_df.randomSplit([0.8, 0.2], seed=42)
print("Training records:", train_df.count())
print("Testing records:", test_df.count())

print("Training model...")
model = pipeline.fit(train_df)
print("Model trained successfully.")

predictions = model.transform(test_df)
evaluator = BinaryClassificationEvaluator(labelCol="will_purchase")
auc = evaluator.evaluate(predictions)
print(f"Model AUC: {auc:.4f}")

predictions.select("customer_id", "will_purchase", "prediction", "probability").show(10)

Training records: 2400782
Testing records: 599218
Training model...
Model trained successfully.
Model AUC: 0.4993
+--------------------+-------------+----------+--------------------+
|         customer_id|will_purchase|prediction|         probability|
+--------------------+-------------+----------+--------------------+
|00001bc8-1ba9-446...|            1|       0.0|[0.65075318854666...|
|00002972-269a-4ed...|            0|       0.0|[0.65159092486973...|
|00003a33-104d-4ce...|            0|       0.0|[0.65161465526578...|
|0000a976-3b2b-4e5...|            0|       0.0|[0.65154742290531...|
|0000fe98-382b-473...|            0|       0.0|[0.65095691958488...|
|0001322b-007a-40a...|            0|       0.0|[0.65057637196974...|
|00017e08-3b25-4c5...|            1|       0.0|[0.65078579161078...|
|0001f304-7626-404...|            0|       0.0|[0.65122281088739...|
|0002695c-0647-4b4...|            0|       0.0|[0.64862241987760...|
|00026a75-67cd-428...|            0|       0.0|[0.64885460

In [8]:
import mysql.connector

conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="",
    database="bigdata_project"
)

# Increase max_allowed_packet
cursor = conn.cursor()
cursor.execute("SET GLOBAL max_allowed_packet=67108864")

insert_query = """
    INSERT INTO customer_predictions
    (customer_id, will_purchase, prediction, probability_no_purchase, probability_purchase)
    VALUES (%s, %s, %s, %s, %s)
"""

batch = []
for row in rows:
    probs = row["probability"]
    batch.append((
        row["customer_id"],
        int(row["will_purchase"]),
        int(row["prediction"]),
        float(probs[0]),
        float(probs[1])
    ))

# Insert in chunks of 5000
chunk_size = 5000
total_inserted = 0
for i in range(0, len(batch), chunk_size):
    chunk = batch[i:i+chunk_size]
    cursor.executemany(insert_query, chunk)
    conn.commit()
    total_inserted += len(chunk)
    print(f"  Inserted {total_inserted:,} rows...")

print(f"\nDone! Total inserted: {total_inserted:,} rows into MySQL.")
cursor.close()
conn.close()

  Inserted 5,000 rows...
  Inserted 10,000 rows...
  Inserted 15,000 rows...
  Inserted 20,000 rows...
  Inserted 25,000 rows...
  Inserted 30,000 rows...
  Inserted 35,000 rows...
  Inserted 40,000 rows...
  Inserted 45,000 rows...
  Inserted 50,000 rows...
  Inserted 55,000 rows...
  Inserted 60,000 rows...
  Inserted 65,000 rows...
  Inserted 70,000 rows...
  Inserted 75,000 rows...
  Inserted 80,000 rows...
  Inserted 85,000 rows...
  Inserted 90,000 rows...
  Inserted 95,000 rows...
  Inserted 100,000 rows...

Done! Total inserted: 100,000 rows into MySQL.
